In [4]:
import pathlib as pl

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]
DATA_ROOT = CONFIG["project_data"]

# custom start
import pandas as pd
import re

sample_re = re.compile("^[A-Z]{2}[0-9]{5}$")

table_ctg = CONFIG["project_repo"].joinpath(
    "annotation", "raw", "20250404_T2T-Y_unplaced_v2.AR.map.tsv"
).resolve(strict=True)
table_scf = CONFIG["project_repo"].joinpath(
    "annotation", "raw", "20250404_T2T-Y_scaffold_v2.AR.map.tsv"
).resolve(strict=True)

orient_map = {
    "+": 1,
    "-": -1
}

def extract_separate_tigs(long_tig_seq):

    sep_pos = re.compile("(\\+_|\\-_)")
    sep_tigs = []
    start = 0
    end = 0
    for mobj in sep_pos.finditer(long_tig_seq):
        begin, stop = mobj.span()
        end = stop - 1
        tig = long_tig_seq[start:end]
        sep_tigs.append(tig)
        start = end + 1
    tig = long_tig_seq[start:]
    sep_tigs.append(tig)
    return sep_tigs


sample_alias_map = {
    "HG002": "NA24385",
    "HG003": "NA24143",
    "HG005": "NA24631",
}

rows = []
for table_file in [table_ctg, table_scf]:
    
    with open(table_file) as table:
        for line in table:
            columns = line.strip().split()
            sample = columns[0].split("_")[0]
            alias = sample_alias_map.get(sample, sample)
            new_seq_name = columns[0]
            if len(columns) == 3:
                if columns[-1] in ["CEN", "shared on globus"]:
                    proc_status = "auto"
                else:
                    proc_status = "manual"
            else:
                proc_status = "auto"
            old_seq_names = columns[1]
            if old_seq_names.count("_") > 1:
                for tig in extract_separate_tigs(columns[1]):
                    orient = tig[-1]
                    orient = orient_map.get(orient, 0)
                    if orient != 0:
                        tig = tig[:-1]
                    rows.append(
                        (sample, alias, tig, new_seq_name, orient, proc_status)
                    )
            else:
                orient = old_seq_names[-1]
                orient = orient_map.get(orient, 0)
                if orient != 0:
                    tig = old_seq_names[:-1]
                else:
                    tig = old_seq_names
                rows.append(
                    (sample, alias, tig, new_seq_name, orient, proc_status)
                )

df = pd.DataFrame.from_records(
    rows,
    columns=["sample", "alias", "old_name", "new_name", "orient", "proc_status"])
df.sort_values(["sample", "new_name"], inplace=True)

out_file = CONFIG["project_repo"].joinpath(
    "annotation", "norm", "20250404_arhie_chry_seqs.v2.tsv"
).resolve()
df.to_csv(out_file, sep="\t", header=True, index=False)

('HG01167', 'HG01167', 'haplotype1-0000001', 'HG01167_chrY', 1, 'manual')


RuntimeError: No active exception to reraise